# Analytic continuation of tabulated transfer functions and reflection coefficients

A flame transfer function, a fuel-injection response, or a boundary reflection coefficient is often known only as **tabulated data** -- a table of complex values on a real frequency grid (measured on a rig, or exported from a high-fidelity simulation).

Nefes consumes that data two ways, and they place very different demands on it.

* The **forced response** and the **Nyquist** stability driver evaluate everything on the **real** frequency axis.
  A grid interpolant (linear, spline, magnitude/phase) is all they need, so a raw table works as-is.
* The **eigenvalue / contour** stability driver searches the **complex** frequency plane (it looks for $\det A(\omega)=0$ off the real axis, where the growth rate lives).
  A real-grid interpolant is *not analytic* -- it cannot be evaluated at a complex frequency -- so a raw table cannot go there.

This notebook builds the bridge: **analytic continuation** of tabulated data.
A table does not determine its own extension off the real axis; one must first say what *kind* of function the response is, and Nefes offers the two physically meaningful answers.

* `fit_impulse_response` -- the **recommended** route whenever the response to a brief disturbance dies out after a finite time, which is true of flames (a finite convective memory) and of compact elements without an internal resonator.
  The result is a finite sum of pure delays: it evaluates at *any* complex frequency and has **no poles anywhere**, so nothing artificial can sit inside a stability search window.
* `rational_fit` -- the alternative for a response with a **genuine resonance** (a cavity damper, a resonant end plate), whose transfer function really has poles; an AAA barycentric-rational fit describes it honestly.

We then:

1. enumerate **every parameter** where Nefes accepts tabulated data (reflection coefficients, the flame FTF, and the fuel-modulation mass response);
2. demonstrate the recommended impulse-response continuation and, for the resonant case, the rational machinery with its helper tools (fit overlay, pole/zero map, diagnostics);
3. run a **stability analysis** on a Rijke tube whose reflection coefficients **and** flame transfer function are both supplied as tabulated data -- continued, then handed to the eigensolver;
4. show that the **Nyquist** method needs no continuation and works **directly** on the same raw tables.

In [ ]:
import warnings

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nefes
from nefes.elements import catalog as cat
from nefes.elements import n_tau_lowpass, tabulated, heat_release_response, mass_flow_response
from nefes.perturbation import PerturbationBC, fit_impulse_response, rational_fit, continuation_warning
from nefes.plotting import use_nefes_theme, plot_fit, plot_pole_map

# Use the bundled theme instead of plotly's default.
use_nefes_theme()

## 1. Where tabulated data enters Nefes

Every frequency-dependent input below accepts the **same four flavours**: a constant, a Python callable `f -> complex`, a raw table `(freqs_hz, values)`, or an analytic continuation of a table (`fit_impulse_response` / `rational_fit`).
The first column is the constructor; the last says which flavours reach the complex-plane eigensolver.

| Quantity | Where it is set | Raw table OK for forced / Nyquist? | Reaches the eigensolver? |
|---|---|---|---|
| **Flame transfer function** $F(f)$ | `heat_release_response(transfer, ref_edge=...)` | yes (`tabulated`) | only if analytic (continued) |
| **Fuel-injection / mass response** $F(f)$ | `mass_flow_response(transfer, ref_edge=...)` | yes (`tabulated`) | only if analytic (continued) |
| **Boundary reflection** $R(f)$ | `PerturbationBC.reflection(R)` | yes (`(f, R)` tuple) | only if analytic (continued) |
| **Boundary impedance** $Z(f)$ | `PerturbationBC.impedance(Z)` | yes (`(f, Z)` tuple) | only if analytic (continued) |

The off-diagonal entropy coupling $R_s(f)$ (`entropy_coupling=`) and any driven-wave `amplitudes` take the same flavours.

The rule is one line: **a raw table is fine on the real axis; continue it -- `fit_impulse_response` for a finite-memory response, `rational_fit` for a resonant one -- to take it into the complex plane.**
The constructors below build identical-shaped objects -- the only difference is whether the carrier is a tuple (real-axis only) or a continuation (analytic).

In [ ]:
# the same FTF, three ways the user might hand it over
f_demo = np.linspace(0.0, 4000.0, 200)
F_true = n_tau_lowpass(1.6, 4.0e-3, 250.0)                # a closed-form reference flame model
samples = np.array([complex(F_true(f)) for f in f_demo])  # ... as if measured on a rig

raw = tabulated(f_demo, samples)                              # real-axis interpolant -> forced / Nyquist
imp = fit_impulse_response(f_demo, samples, duration=12e-3)   # recommended continuation -> + eigensolver
fit = rational_fit(f_demo, samples)                          # resonant-response continuation
const = 1.2                                                  # a frequency-flat gain

print("raw  :", raw)
print("imp  :", imp, "  analytic =", imp.analytic)
print("fit  :", fit, "  analytic =", fit.analytic)
print("\nflame source from a fit  :", heat_release_response(fit, ref_edge=1))
print("mass  source from a fit  :", mass_flow_response(fit, ref_edge=1))
print("reflection BC from a fit :", PerturbationBC.reflection(fit).kind)

## 2. The recommended continuation: `fit_impulse_response`

`fit_impulse_response(freqs, values, duration=...)` assumes the response to a brief disturbance dies out within `duration` seconds and recovers the impulse response behind the samples: a least squares with a penalty on its second difference, so it follows the trend of the data rather than every measurement wiggle.
The result is a `FiniteImpulseResponse`, a finite sum of pure delays -- *entire*, no poles anywhere -- so its value at a complex frequency is exact analytic continuation, with nothing artificial for a stability search to trip over.

The one physical input is `duration`, the memory length: a few times the largest transport delay in the data (readable as the slope of the phase).
The sample spacing defaults to $1/(2 f_{\max})$, so the fit carries no frequency content the data cannot constrain; here the gain has not fully rolled off by the top of the band, so we halve it (exactly at the resolvable limit every basis term turns real and the band edge is matched only loosely).
Here the flame forgets after its $4\;\mathrm{ms}$ lag plus a $\sim 1\;\mathrm{ms}$ roll-off memory, so $12\;\mathrm{ms}$ is generous.

In [ ]:
ftf_imp = fit_impulse_response(f_demo, samples, duration=12e-3, dt=0.0625e-3)
print("on-grid quality      : rms misfit = %.2e   max = %.2e" % (ftf_imp.rms_misfit, ftf_imp.max_misfit))

# off the real axis: the continuation matches the closed form it was sampled from
z = 127.0 - 104.0 / (2 * np.pi) * 1j    # a lightly-unstable complex frequency [Hz]
err = abs(complex(ftf_imp(z)) - complex(F_true(z)))
print("off-axis |error| at z:", "%.2e" % err, " (truncation of the response tail -- there are no poles to avoid)")

fig = go.Figure()
fig.add_bar(x=ftf_imp.lags * 1e3, y=ftf_imp.h, name="fitted impulse response")
fig.update_layout(title="The impulse response behind the sampled FTF",
                  xaxis_title="lag [ms]", yaxis_title="coefficient", height=320)
fig.show()

## 3. The rational alternative: `rational_fit`

For a response with a **genuine resonance**, whose ringing outlasts any reasonable memory length, `rational_fit(freqs, values)` fits the samples with a barycentric **rational** function via the AAA algorithm (Nakatsukasa--Sete--Trefethen 2018).
A rational function is analytic everywhere except at its poles, so it continues off the real axis -- at the price that a fit of noisy data can place *artificial* poles near the search region, which is why this is the second choice and why the diagnostics below exist.

Below we take a "measured" FTF (the closed form sampled on a coarse grid) and continue it.
`plot_fit` overlays the continuation (line) on the data (markers).

In [ ]:
f_meas = np.linspace(0.0, 4000.0, 90)                        # a coarse "measured" grid
F_meas = np.array([complex(F_true(f)) for f in f_meas])      # the measured FTF samples

ftf_fit = rational_fit(f_meas, F_meas, delay="auto")         # the analytic continuation

print("support points :", ftf_fit.n_terms)
print("fitted delay   : %.3f ms" % (ftf_fit.delay * 1e3))
print("max fit error  : %.2e" % ftf_fit.max_error())
plot_fit(ftf_fit, extend=0.15, phase="deg",
         title="Measured flame FTF and its analytic continuation").show()

### The continuation goes where a raw table cannot

The whole point: evaluate at a **complex** frequency.
A growing mode at $f \approx 130\,\mathrm{Hz}$, growth $\sigma = 100\,\mathrm{s}^{-1}$ sits at the complex frequency $f - i\,\sigma/2\pi$.
The raw table refuses it; the continuation returns a value.

In [ ]:
z = 127.0 - 104.0 / (2 * np.pi) * 1j   # a lightly-unstable complex frequency [Hz] (near the Rijke mode below)

try:
    raw(z)
except Exception as e:
    print("raw table  F(z) :", type(e).__name__, "--", str(e).split(".")[0])

print("continuation F(z) :", complex(rational_fit(f_demo, samples, delay='auto')(z)))
print("closed form  F(z) :", complex(F_true(z)))

### Pole / zero map: is the continuation trustworthy where we search?

A rational fit is reliable in the strip around the real axis spanned by the data; far off-axis its poles take over.
`plot_pole_map` shows the fit's poles ($\times$) and zeros ($\circ$) in the same *(frequency, growth)* plane as the eigenvalue spectrum, with the search window shaded.
A well-behaved fit keeps every pole **well away** from the window we sweep, so the continuation is trustworthy there.
`poles_in_region` reports any that intrude (and `continuation_warning` raises a warning).

In [ ]:
band, gband = (40.0, 320.0), (-300.0, 300.0)   # the stability search window

bad = ftf_fit.poles_in_region(band, gband)
print("FTF poles (Hz):", np.round(ftf_fit.poles, 1))
print("poles inside the search window:", bad.size, "(0 = the fit is safe to search there)")
plot_pole_map(ftf_fit, freq_band=band, growth_band=gband,
              title="FTF continuation pole / zero map").show()

### Delay extraction and noise control -- the robustness knobs

A flame FTF is **delay-dominated**: the pure lag $e^{-i 2\pi f \tau}$ is entire but not rational, so AAA spends poles approximating it.
Passing `delay="auto"` (or an explicit $\tau$) peels the lag off before the fit and re-applies it analytically, leaving a low-order rational with poles tucked safely off-axis.
For noisy data, `rtol` stops the fit at the noise floor instead of chasing every wiggle to machine precision (which would seed poles on the real axis).

In [ ]:
# (a) delay extraction collapses a delay-dominated FTF to a handful of poles
print("delay extraction (clean, delay-dominated FTF):")
print(f"   delay=None : {rational_fit(f_meas, F_meas).n_terms:3d} support points")
g_auto = rational_fit(f_meas, F_meas, delay='auto')
print(f"   delay=auto : {g_auto.n_terms:3d} support points  ({g_auto.delay * 1e3:.2f} ms lag peeled off)")

# (b) rtol stops a *noisy* fit at the noise floor instead of chasing every wiggle
rng = np.random.default_rng(1)
F_noisy = F_meas + 5e-3 * (rng.standard_normal(f_meas.size) + 1j * rng.standard_normal(f_meas.size))
greedy = rational_fit(f_meas, F_noisy, delay='auto')              # rtol=None -> chases the noise
robust = rational_fit(f_meas, F_noisy, delay='auto', rtol=1e-2)   # rtol ~ noise floor -> smooth
print("\nover-fitting control (noisy data):")
print(f"   rtol=None  : {greedy.n_terms:3d} support points, max_err {greedy.max_error():.1e}")
print(f"   rtol=1e-2  : {robust.n_terms:3d} support points, max_err {robust.max_error():.1e}")

## 4. The other tabulated surfaces: reflection coefficients and a mass response

The continuation makes no assumption about *what* the data represents -- it is just a complex function of frequency.
Here are the two boundary reflection coefficients and a fuel-injector mass response we will use, each tabulated and continued.

In [ ]:
# closed-form references (stand-ins for measured/exported data)
def R_in_true(f):    # near-rigid inlet with a small end correction and some acoustic loss
    return 0.85 * np.exp(-1j * 2 * np.pi * f * 2.0e-5)

def R_out_true(f):   # lossy open end with a (high) radiation resonance
    s = 1j * f / 950.0
    return -0.6 / (1.0 + 2.0 * 0.5 * s + s * s)

def Finj_true(f):    # a velocity-driven fuel injector: low-pass with its own short lag
    return 0.6 * np.exp(-1j * 2 * np.pi * f * 1.5e-3) / (1.0 + 1j * f / 250.0)

f_R = np.linspace(0.0, 4000.0, 70)
fit_in  = rational_fit(f_R, np.array([complex(R_in_true(f))  for f in f_R]), delay="auto")
fit_out = rational_fit(f_R, np.array([complex(R_out_true(f)) for f in f_R]))
fit_inj = rational_fit(f_R, np.array([complex(Finj_true(f))  for f in f_R]), delay="auto")

for name, g in [("R_in", fit_in), ("R_out", fit_out), ("F_inj", fit_inj)]:
    print(f"{name:6s}: {g.n_terms} terms, max_err {g.max_error():.1e}, "
          f"{g.poles_in_region(band, gband).size} pole(s) in window")

    print("\nfuel-injection response is analytic -> usable for stability:",
          mass_flow_response(fit_inj, ref_edge=1).analytic)
plot_fit(fit_out, extend=0.1, phase="deg",
         title="Outlet reflection coefficient R_out(f): data vs continuation").show()

The fuel-injection response would drive a `mass_source` element via `mass_flow_response(fit_inj, ref_edge=...)` exactly as the FTF drives the flame; we keep the stability case below to a single flame source for clarity.

## 5. Rijke-tube stability with tabulated reflections **and** a tabulated FTF

The classic self-excited combustor: a cold-air inlet, a compact heat-release flame between two ducts, and an outlet.
We feed it **everything as tabulated data** -- the inlet reflection $R_\mathrm{in}(f)$, the outlet reflection $R_\mathrm{out}(f)$, *and* the flame transfer function $F(f)$ -- continue each (the finite-memory flame with `fit_impulse_response`, the reflections with `rational_fit`), and hand the continuations to the **eigensolver**.

![Network topology](analytic_continuation_topology.png)

The inlet carries $R_\mathrm{in}(f)$ and the outlet $R_\mathrm{out}(f)$; the flame between the cold and hot ducts carries the transfer function $F(f)$.

In [ ]:
R_AIR, GAMMA = 287.0, 1.4
CP = GAMMA * R_AIR / (GAMMA - 1.0)
AREA, L1, L2 = 0.01, 0.6, 0.4          # tube section and cold / hot duct lengths
MDOT, T_IN, P_OUT, DT = 0.005, 300.0, 1.0e5, 400.0

# the flame FTF as a measured table, sampled out to where its gain has rolled off;
# a flame is finite-memory, so the impulse-response fit is the natural continuation
f_ftf = np.linspace(0.0, 4000.0, 360)
F_ftf_data = np.array([complex(F_true(f)) for f in f_ftf])
fit_ftf = fit_impulse_response(f_ftf, F_ftf_data, duration=12e-3, dt=0.0625e-3)


def rijke(ftf_transfer, bc_in, bc_out):
    """Build and mean-solve the Rijke tube with the given flame source and boundary closures."""
    nodes = [
        cat.mass_flow_inlet(MDOT, T_IN, perturbation_bc=bc_in),
        cat.duct(L1, name="cold duct"),
        cat.heat_release_flame(MDOT * CP * DT, dynamic_source=heat_release_response(ftf_transfer, ref_edge=1)),
        cat.duct(L2, name="hot duct"),
        cat.pressure_outlet(P_OUT, perturbation_bc=bc_out),
    ]
    edges = [(0, 1, AREA), (1, 2, AREA), (2, 3, AREA), (3, 4, AREA)]
    net = nefes.Network(nefes.perfect_gas(R_AIR, GAMMA),
                        nodes=nodes, edges=edges)
    sol = net.solve()
    assert sol.converged, "mean flow did not converge"
    return sol


# all three inputs continued from their tables
bc_in = PerturbationBC.reflection(fit_in)
bc_out = PerturbationBC.reflection(fit_out)
sol = rijke(fit_ftf, bc_in, bc_out)
print("mean flow converged; flame heat release Q = %.0f W" % (MDOT * CP * DT))

In [ ]:
def modes(sol):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")        # certification / band-edge notes
        return sol.eigenmodes(freq_band=band, growth_band=gband, isentropic=True)

# passive baseline (flame gain zeroed) vs the active flame
sol_passive = rijke(n_tau_lowpass(0.0, 0.0, 1.0), bc_in, bc_out)
res_passive = modes(sol_passive)
res_active = modes(sol)

print("passive flame:")
for m in res_passive.summary():
    print(f"   f = {m['freq_hz']:7.2f} Hz   growth = {m['growth_rate']:+8.2f} 1/s")
print("active flame (tabulated FTF + tabulated reflections, continued):")
for m in res_active.summary():
    flag = "   <-- UNSTABLE" if m["unstable"] else ""
    print(f"   f = {m['freq_hz']:7.2f} Hz   growth = {m['growth_rate']:+8.2f} 1/s{flag}")

res_active.plot_spectrum(title="Rijke spectrum -- all inputs tabulated, then continued").show()

### Validation: the continuation reproduces the closed-form spectrum

The continuations agree with their closed forms on the real axis -- the rational fits to ~$10^{-13}$, the impulse-response fit to the ~$10^{-6}$ truncation of the flame's exponential roll-off tail -- so the eigenvalues they produce must match the ones from feeding the closed-form callables straight in.
This is the check that the tabulated route adds no error of its own.

In [ ]:
sol_cf = rijke(F_true, PerturbationBC.reflection(R_in_true), PerturbationBC.reflection(R_out_true))
res_cf = modes(sol_cf)

f_tab, g_tab = max(zip(res_active.freqs, res_active.growth_rates), key=lambda m: m[1])
f_cf, g_cf = max(zip(res_cf.freqs, res_cf.growth_rates), key=lambda m: m[1])
print("most-unstable mode")
print(f"   closed-form inputs : f = {f_cf:.4f} Hz,  growth = {g_cf:+.4f} 1/s")
print(f"   tabulated inputs   : f = {f_tab:.4f} Hz,  growth = {g_tab:+.4f} 1/s")
print(f"   difference         : df = {abs(f_tab - f_cf):.2e} Hz,  dg = {abs(g_tab - g_cf):.2e} 1/s")

## 6. Nyquist on the real axis: verdict, onset, and the off-axis mode

The eigensolver of Section 5 found the growing mode by searching the **complex** plane -- which is exactly why it needed every input continued.
The **Nyquist** driver answers the same question from a sweep on the **real** frequency axis alone, so it never evaluates the inputs off-axis.
A single real-frequency sweep carries three levels of information, in increasing detail:

1. the **unstable-mode count** -- how many times the return-ratio locus encircles the critical point (the argument principle).
   It is a robust integer, and it reads straight off the **raw** tables: the kinks of a piecewise-linear table do not move the winding.
2. the **onset frequencies** -- where the locus skims the critical point ($|D(\omega)| = |\det(I - L)|$ dips), i.e. the frequencies of the *marginal* modes.
3. the **off-axis frequency and growth of every mode** -- by fitting a rational function to the scalar determinant $D(\omega)$ along the real axis and reading off its complex **zeros** (`mode_estimates`).
   This is the eigensolver's own information, recovered with no off-axis evaluation of the operator.

Levels 1--2 need only the raw real-axis samples.
Level 3 *fits* $D(\omega)$, so it wants a **smooth** determinant: feed the `rational_fit` continuations (real-axis-exact and smooth) or a dense table -- a coarse piecewise-linear table would scatter the fit with a pole at every kink.
So the continuation framework is not only the eigensolver's key; it is also what sharpens the Nyquist sweep into a mode locator.

In [ ]:
# levels 1-2: the verdict straight from the RAW tables -- no continuation, real axis only
raw_ftf = tabulated(f_ftf, np.array([complex(F_true(f)) for f in f_ftf]))
raw_bc_in = PerturbationBC.reflection((f_R, np.array([complex(R_in_true(f)) for f in f_R])))
raw_bc_out = PerturbationBC.reflection((f_R, np.array([complex(R_out_true(f)) for f in f_R])))
sol_raw = rijke(raw_ftf, raw_bc_in, raw_bc_out)

freqs_ny = np.linspace(0.0, 1800.0, 1000)        # real-axis sweep, out to where the locus is quiet
ny_raw = sol_raw.nyquist_stability(freqs_ny, isentropic=True)

print(f"n_unstable = {ny_raw.n_unstable}    stable = {ny_raw.stable}    margin min|D| = {ny_raw.margin:.3f}")
print("onset (crossing) frequencies:",
      [f"{c['freq_hz']:.0f} Hz (|D|={c['abs_D']:.2f})" for c in ny_raw.crossings()])
print("\nthe count says one mode is unstable -- but the crossing is the near-marginal frequency, not the\n"
      "growing mode: a strongly growing mode sits off the real axis and leaves no dip in |D|. Hence level 3.")

In [ ]:
# the Nyquist locus: the return ratio L(omega) and its conjugate mirror, with the critical point +1
L = np.atleast_1d(ny_raw.L).ravel() if ny_raw.L.ndim == 1 else ny_raw.L[:, 0, 0]
loc = np.concatenate([np.conj(L[::-1]), L])    # close the contour over +/- frequency

fig = make_subplots(rows=1, cols=2, subplot_titles=("Nyquist locus of the return ratio L(\u03c9)", "|D(\u03c9)| = |det(I \u2212 L)|"))
fig.add_scatter(x=loc.real, y=loc.imag, mode="lines", line_color="#2563eb", name="L(\u03c9)", row=1, col=1)
fig.add_scatter(x=[1.0], y=[0.0], mode="markers+text", text=["+1"], textposition="top right",
                marker=dict(size=10, color="#ef4444", symbol="x"), name="critical point", row=1, col=1)
fig.add_scatter(x=ny_raw.freqs, y=np.abs(ny_raw.D), mode="lines", line_color="#10b981", name="|D|", row=1, col=2)
for c in ny_raw.crossings():
    fig.add_scatter(x=[c["freq_hz"]], y=[c["abs_D"]], mode="markers", marker=dict(size=9, color="#ef4444"),
                    name=f"onset {c['freq_hz']:.0f} Hz", row=1, col=2)
fig.update_xaxes(title_text="Re L", row=1, col=1)
fig.update_yaxes(title_text="Im L", scaleanchor="x", scaleratio=1.0, row=1, col=1)
fig.update_xaxes(title_text="frequency [Hz]", row=1, col=2)
fig.update_yaxes(title_text="|D|", row=1, col=2)
fig.update_layout(title="Nyquist test on the raw tabulated data", showlegend=False, height=420)
fig.show()

### The growing mode's frequency and growth, from the same real-axis sweep

The count says *one* mode is unstable; the crossing gave only the near-marginal frequency.
To place the growing mode we fit a rational to $D(\omega)$ along the real axis and read its complex zeros (`mode_estimates`).
That fit wants a smooth determinant, so we run the same real-axis sweep on the **continuations** (Section 4's `rational_fit`s -- exact on the real axis, and smooth).
No complex-plane evaluation of the operator takes place: only the *scalar* $D(\omega)$, sampled on the real axis, is continued.

In [ ]:
# level 3: feed the smooth continuations to the same real-axis sweep, fit D(omega), read its zeros
ny_cont = sol.nyquist_stability(freqs_ny, isentropic=True)   # sol carries the continued inputs (Section 5)
est = ny_cont.mode_estimates(unstable_only=True)

print("Nyquist off-axis estimate (rational fit of D on the real axis):")
for m in est:
    print(f"   f = {m['freq_hz']:7.2f} Hz   growth = {m['growth_rate']:+8.2f} 1/s")
print(f"\neigensolver (complex contour)        : f = {f_tab:7.2f} Hz   growth = {g_tab:+8.2f} 1/s")
m0 = min(est, key=lambda e: abs(e['freq_hz'] - f_tab))
print(f"Nyquist vs eigensolver               : df = {abs(m0['freq_hz'] - f_tab):.2e} Hz   "
      f"dg = {abs(m0['growth_rate'] - g_tab):.2e} 1/s")
assert ny_raw.n_unstable == 1 and m0["unstable"], "one growing mode, located both ways"
assert ny_cont.passive_assumption_ok, "A_0 passive -> the count is the absolute unstable-mode number"

### Two roads to the same mode

* the **eigensolver** continued every input and searched the complex plane $\rightarrow f \approx 127\,\mathrm{Hz}$, growth $\approx +104\,\mathrm{s}^{-1}$;
* **Nyquist** swept the real axis $\rightarrow$ the same count, the onset frequency, and -- from the rational fit of $D$ -- the same off-axis mode.

On this purely acoustic tube both drivers are valid, so they cross-check each other to $\sim 10^{-3}$.
The reason the real-axis route exists at all is the regime where the eigensolver **cannot** run: once a convected entropy or equivalence-ratio wave is retained (`isentropic=False`, a choked-nozzle or reacting outlet), the contour integrand overflows and the spectrum fills with near-marginal convected modes.
There the very same real-axis machinery -- count, onset, and `mode_estimates` -- is the only one that survives (see `equivalence_ratio_instability.ipynb` and `indirect_noise_instability.ipynb`).

## Summary

* Tabulated transfer functions and reflection coefficients reach Nefes through `heat_release_response`, `mass_flow_response`, `PerturbationBC.reflection`, and `PerturbationBC.impedance` -- each accepting a constant, a callable, a raw table, or an analytic continuation.
* The continuation is chosen by the physics of the response.
  `fit_impulse_response` is the recommended route for a **finite-memory** response (a flame, a compact element): the result is a finite sum of delays with **no poles anywhere**, so nothing artificial can sit inside a stability search window; the one physical input is the memory length.
  `rational_fit` is the choice for a **resonant** response, whose transfer function genuinely has poles; helpers `plot_fit`, `plot_pole_map`, `poles_in_region`, and `continuation_warning` make the fit and its validity easy to inspect, and `delay="auto"` and `rtol` are the robustness knobs.
* A Rijke tube with **all** of its frequency-dependent inputs tabulated reproduces the closed-form instability once continued (flame by impulse response, reflections by rational fit).
  The **Nyquist** driver reads the verdict and the onset frequency directly off the raw real-axis tables, and -- by fitting the scalar determinant $D(\omega)$ along that same sweep (`mode_estimates`) -- recovers the growing mode's frequency **and** growth too, matching the eigensolver to $\sim 10^{-3}$.

## Export for Nemo

The converged Rijke mean flow is available as `sol` (a `Solution`).
Save it to a UI-readable YAML with `sol.to_yaml(path)` (topology plus the embedded mean-flow fields), then open the file in Nemo.

In [ ]:
import os
import tempfile

_out = os.path.join(tempfile.mkdtemp(), "analytic_continuation.yaml")
sol.to_yaml(_out)  # topology + the embedded mean-flow results
print("saved case:", _out)